### Thử NER với qwen2.5 local llm
### File này sẽ load thử vài file txt của text data để xem qwen có nhận diện các trường trong output hay không

In [1]:
from langchain_ollama import ChatOllama
import json
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import TextLoader
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


C:\Users\ACER\AppData\Local\Temp\ipykernel_13112\2520606264.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
# Load 1.txt từ test data.
data_path = Path("./data/test_data")

loader = TextLoader("../../data/test_data/2.txt", encoding="utf-8")
docs = loader.load()
text = docs[0].page_content
print(f"doc: {text}")

doc: 1.  Tiền sử bệnh nội khoa
    Các bệnh lý mãn tính: nghi ngờ xơ gan do rượu
    Các yếu tố nguy cơ liên quan: Có tiền sử sử dụng rượu bia

2.  Tiền sử bệnh hiện tại
    Triệu chứng hiện tại: hội chứng não gan
    Các sự kiện trước khi nhập viện
    - đến bệnh viện  với hội chứng não gan
    - Bệnh nhân được chuyển viện trong tình trạng hội chứng não gan tiến triển, ý thức suy giảm
    - Bệnh nhân nhập viện vào ngày w tiến triển hội chứng não gan
    - không thể tự chăm sóc bản thân tại nhà

3.  Đánh giá tại bệnh viện
    Các thủ thuật đã thực hiện: đặt shunt dẫn lưu tĩnh mạch cửa qua da


In [ ]:
template = """
Bạn là hệ thống trích xuất khái niệm y tế từ văn bản bệnh án tiếng Việt.

Nhiệm vụ:
Từ MỘT văn bản đầu vào duy nhất, hãy phát hiện các cụm từ y tế xuất hiện nguyên văn trong văn bản và phân loại chúng vào đúng type.

Chỉ được dùng các type sau:
1. TRIỆU_CHỨNG
2. TÊN_XÉT_NGHIỆM
3. KẾT_QUẢ_XÉT_NGHIỆM
4. CHẨN_ĐOÁN
5. THUỐC

Bạn chỉ được trả về JSON hợp lệ.
Không giải thích.
Không markdown.
Không dùng ```json.

Output phải là list các dictionary theo format:

[
  {{
    "text": "chuỗi y tế xuất hiện nguyên văn trong input",
    "type": "một trong 5 type hợp lệ"
  }}
]

Quy tắc bắt buộc:
- "text" phải được copy chính xác từ input.
- Không tự sửa lỗi chính tả.
- Không tự chuẩn hóa thuật ngữ.
- Không tự dịch sang tiếng Anh.
- Không thêm field khác ngoài "text" và "type".
- Không trả về "position". Position sẽ được xử lý bằng code sau.
- Nếu cùng một cụm xuất hiện nhiều lần trong input, hãy trả về nhiều object tương ứng.
- Nếu không có khái niệm y tế nào, trả về [].
- Không trích xuất thủ thuật, tên khoa/phòng, ngày tháng, thông tin hành chính nếu chúng không thuộc 5 type trên.

Định nghĩa type:

1. THUỐC
Dùng cho tên thuốc hoặc cụm thuốc có liều lượng/cách dùng đi kèm.
Ví dụ:
- "amlodipine 10 mg po daily"
- "aspirin 81 mg po daily"
- "metoprolol succinate xl 50 mg po daily"
- "nystatin oral suspension 5 ml po qid:prn"

2. TRIỆU_CHỨNG
Dùng cho biểu hiện lâm sàng, dấu hiệu, triệu chứng, cảm giác bất thường của bệnh nhân.
Ví dụ:
- "ho"
- "sốt"
- "khó thở"
- "đau bụng"
- "ý thức suy giảm"
- "đau nhức"

Phân biệt:
- "ý thức suy giảm" là TRIỆU_CHỨNG vì mô tả biểu hiện của bệnh nhân.
- "hội chứng não gan" là CHẨN_ĐOÁN vì là tên hội chứng/bệnh lý.

3. CHẨN_ĐOÁN
Dùng cho tên bệnh, hội chứng, tình trạng bệnh lý đã được chẩn đoán hoặc nghi ngờ.
Ví dụ:
- "tăng huyết áp"
- "đái tháo đường"
- "viêm phổi"
- "suy tim"
- "xơ gan do rượu"
- "hội chứng não gan"

Phân biệt:
- "khó thở" là TRIỆU_CHỨNG.
- "viêm phổi" là CHẨN_ĐOÁN.
- "đau bụng" là TRIỆU_CHỨNG.
- "xơ gan do rượu" là CHẨN_ĐOÁN.

4. TÊN_XÉT_NGHIỆM
Dùng cho tên xét nghiệm, tên chỉ số xét nghiệm, marker, thông số cận lâm sàng.
Ví dụ:
- "WBC"
- "NEUT%"
- "CRP"
- "creatinin"
- "glucose máu"
- "AST"
- "ALT"

5. KẾT_QUẢ_XÉT_NGHIỆM
Dùng cho giá trị hoặc mô tả kết quả xét nghiệm.
Ví dụ:
- "14,43"
- "76,4"
- "dương tính"
- "âm tính"
- "tăng"
- "giảm"

Phân biệt:
- Trong "WBC 14,43", "WBC" là TÊN_XÉT_NGHIỆM, "14,43" là KẾT_QUẢ_XÉT_NGHIỆM.
- Trong "CRP tăng", "CRP" là TÊN_XÉT_NGHIỆM, "tăng" là KẾT_QUẢ_XÉT_NGHIỆM.

Ví dụ 1:

Input:
"Danh sách thuốc trước nhập viện chính xác và đầy đủ. 1. amlodipine 10 mg po daily 2. aspirin 81 mg po daily 3. metoprolol succinate xl 50 mg po daily 4. guaifenesin ml po q6h:prn điều trị ho 5. nystatin oral suspension 5 ml po qid:prn điều trị đau nhức."

Output:
[
  {{
    "text": "amlodipine 10 mg po daily",
    "type": "THUỐC"
  }},
  {{
    "text": "aspirin 81 mg po daily",
    "type": "THUỐC"
  }},
  {{
    "text": "metoprolol succinate xl 50 mg po daily",
    "type": "THUỐC"
  }},
  {{
    "text": "guaifenesin ml po q6h:prn",
    "type": "THUỐC"
  }},
  {{
    "text": "ho",
    "type": "TRIỆU_CHỨNG"
  }},
  {{
    "text": "nystatin oral suspension 5 ml po qid:prn",
    "type": "THUỐC"
  }},
  {{
    "text": "đau nhức",
    "type": "TRIỆU_CHỨNG"
  }}
]

Ví dụ 2:

Input:
"Các bệnh lý mãn tính: nghi ngờ xơ gan do rượu. Triệu chứng hiện tại: hội chứng não gan."

Output:
[
  {{
    "text": "xơ gan do rượu",
    "type": "CHẨN_ĐOÁN"
  }},
  {{
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN"
  }}
]

Ví dụ 3:

Input:
"Bệnh nhân được chuyển viện trong tình trạng hội chứng não gan tiến triển, ý thức suy giảm."

Output:
[
  {{
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN"
  }},
  {{
    "text": "ý thức suy giảm",
    "type": "TRIỆU_CHỨNG"
  }}
]

Ví dụ 4:

Input:
"Kết quả xét nghiệm ghi nhận WBC 14,43, NEUT% 76,4 và CRP tăng."

Output:
[
  {{
    "text": "WBC",
    "type": "TÊN_XÉT_NGHIỆM"
  }},
  {{
    "text": "14,43",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  }},
  {{
    "text": "NEUT%",
    "type": "TÊN_XÉT_NGHIỆM"
  }},
  {{
    "text": "76,4",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  }},
  {{
    "text": "CRP",
    "type": "TÊN_XÉT_NGHIỆM"
  }},
  {{
    "text": "tăng",
    "type": "KẾT_QUẢ_XÉT_NGHIỆM"
  }}
]

Bây giờ hãy xử lý văn bản sau.

Input:
{input_text}

Output:
"""
qwen = ChatOllama(model="qwen2.5:7b", temperature=0)
embedding_prompt = ChatPromptTemplate.from_template(template)
generate_emd_json = (
     embedding_prompt
    | qwen
    | StrOutputParser()
)
json_result = generate_emd_json.invoke({
    "input_text": text
})
print(f"Result: {json_result}")


Result: [
  {
    "text": "xơ gan do rượu",
    "type": "CHẨN_ĐOÁN"
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁΝ"
  },
  {
    "text": "ý thức suy giảm",
    "type": "TRIỆU_CHỨNG"
  }
]


### So với kết quả của claude sonnet 5 medium (cùng prompt và cùng input)
```json
[
  {
    "text": "xơ gan do rượu",
    "type": "CHẨN_ĐOÁN",
    "position": [61, 75]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [191, 208]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [270, 287]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [338, 355]
  },
  {
    "text": "ý thức suy giảm",
    "type": "TRIỆU_CHỨNG",
    "position": [368, 383]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [432, 449]
  }
]
```


### So với kết quả của chatgpt5.5 high (cùng prompt và cùng input)
```json
[
{
"text": "xơ gan do rượu",
"type": "CHẨN_ĐOÁN",
"position": [61, 75]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [191, 208]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [270, 287]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [338, 355]
},
{
"text": "ý thức suy giảm",
"type": "TRIỆU_CHỨNG",
"position": [368, 383]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [432, 449]
}
]
```


# Đánh giá
Đầu tiên, có thể thấy Claude và GPT đều phân loại trường text tốt hơn Qwen, trong khi Qwen bỏ sót khá nhiều. Tiếp theo, đối với trường position, kết quả của Claude và GPT trùng khớp với nhau, còn Qwen thì hầu như phân loại sai. Cuối cùng, ở trường type, cả ba mô hình đều cần được cải thiện thêm.

# Hướng phát triển
Thử cái model local pretrained cho y khoa. <br>
Fine tune model bằng custom dataset mới.